In [13]:
import json

import src.travel_agent.config  # loads .env and sets API key

from agents import Runner, trace

from src.travel_agent.agents.intake_agent import intake_agent
from src.travel_agent.agents.visa_agent import visa_agent
from src.travel_agent.agents.activities_agent import activities_agent
from src.travel_agent.agents.transport_agent import transport_agent

from src.travel_agent.tools.transport_tool import search_flights_raw

## 1. Intake: multi-turn chat


In [14]:
history = []

async def say(msg):
    global history
    result = await Runner.run(intake_agent, history + [{'role': 'user', 'content': msg}])
    history = result.to_input_list()
    out = result.final_output
    print('missing:', out.missing_fields)
    print('assistant:', out.next_question)
    return out

out = await say('I want to visit Japan with 2 friends on Oct 26 for 7 days, budget 6000 USD')

missing: ['trip_type', 'passport_country', 'origin', 'interests']
assistant: Is this trip for leisure or business?


In [15]:
out = await say('I hold a French passport, flying from Miami. Food and temples for vacation.')

missing: ['trip_type']
assistant: Is this trip for leisure or business?


In [16]:
trip = out.trip
print(trip.model_dump_json(indent=2))
print('complete:', out.is_complete)

{
  "trip_type": null,
  "passport_country": "France",
  "origin": "Miami",
  "destination": "Japan",
  "start_date": "2026-10-26",
  "end_date": "2026-11-01",
  "num_days": 7,
  "travelers": 3,
  "budget_usd": 6000.0,
  "interests": [
    "food",
    "temples"
  ],
  "meeting_address": null,
  "needs_car_rental": null
}
complete: False


## 2. Visa agent


In [17]:
with trace(""):
    r = await Runner.run(visa_agent, f'Passport: {trip.passport_country}. Destination: {trip.destination}.')
print(r.final_output.model_dump_json(indent=2))

{
  "passport": "FRA",
  "destination": "JPN",
  "requirement": "visa_free",
  "allowed_days": 90,
  "summary": "French passport holders can enter Japan without a visa for up to 90 days. This makes travel to Japan convenient and visa-free for short stays.",
  "source": "orizn_api"
}


## 3. Activities agent


In [18]:
prompt = (
    f'Destination: {trip.destination}. Dates: {trip.start_date} to {trip.end_date}. '
    f'Travelers: {trip.travelers}. Budget USD: {trip.budget_usd}. '
    f'Interests: {trip.interests}. Trip type: {trip.trip_type}.'
)
with trace('Day4 activities'):
    r = await Runner.run(activities_agent, prompt)
plan = r.final_output
print(plan.weather_summary)
for a in plan.activities:
    print('-', a.name, '|', a.category, '|', a.estimated_cost_usd)

Typical seasonal weather for late October to early November in Japan, generally mild and pleasant, suitable for outdoor activities and sightseeing.
- Kyoto Temple and Historical Sights Tour | temples | None
- Kyoto Curry Workshop at Kyoto Curry Seisakusho Kariru | food | 30.0
- Visit to the Kyoto City History Museum | temples/culture | 10.0
- Dinner at Marumaru Seimen (Kyoto Noodles) | food | 20.0
- Visit to Nara's Historic Temples and Sights | temples | None
- Lunch at Shokudo Kanbashi in Nara | food | 15.0


## 4. Transport agent

Search round-trip economy flights for three adult travelers.
Use Duffel test mode with a labeled JSON mock fallback.

For this demo, explicitly select MIA and HND as the flight endpoints.
These choices do not modify the original Intake request.

In [1]:
assert out.is_complete, "Complete intake before running transport."
assert trip.start_date is not None
assert trip.end_date is not None
assert trip.travelers is not None

transport_trip = trip.model_copy(
    update={
        "origin": "MIA",
        "destination": "HND",
    }
)

print(transport_trip.model_dump_json(indent=2))

NameError: name 'out' is not defined

In [ ]:
raw_transport = await search_flights_raw(
    origin=transport_trip.origin,
    destination=transport_trip.destination,
    start_date=transport_trip.start_date,
    end_date=transport_trip.end_date,
    travelers=transport_trip.travelers,
)

print("Source:", raw_transport["source"])
print(json.dumps(raw_transport, indent=2))

Source: duffel_test
{
  "source": "duffel_test",
  "options": [
    {
      "mode": "flight",
      "description": "DUFFEL TEST SAMPLE: Duffel Airways: MIA \u2192 HND; departure 2026-10-26T19:21:00; arrival 2026-10-28T00:55:00 | Duffel Airways: HND \u2192 MIA; departure 2026-11-01T00:00:00; arrival 2026-11-01T02:34:00. Offer ID: off_0000BB5ZKAeML6azQHzuCK",
      "cost_usd_total": 2634.93,
      "is_estimate": true
    },
    {
      "mode": "flight",
      "description": "DUFFEL TEST SAMPLE: British Airways: MIA \u2192 HND; departure 2026-10-26T19:21:00; arrival 2026-10-28T00:55:00 | British Airways: HND \u2192 MIA; departure 2026-11-01T00:00:00; arrival 2026-11-01T02:34:00. Offer ID: off_0000BB5ZKAeiJmsZROABkf",
      "cost_usd_total": 2639.04,
      "is_estimate": true
    },
    {
      "mode": "flight",
      "description": "DUFFEL TEST SAMPLE: Iberia: MIA \u2192 HND; departure 2026-10-26T19:21:00; arrival 2026-10-28T00:55:00 | Iberia: HND \u2192 MIA; departure 2026-11-01T00:00:00

In [ ]:
with trace("Transport agent"):
    transport_result = await Runner.run(
        transport_agent,
        transport_trip.model_dump_json(),
    )

transport_plan = transport_result.final_output

print(transport_plan.model_dump_json(indent=2))

{
  "options": [
    {
      "mode": "flight",
      "description": "DUFFEL TEST SAMPLE: British Airways: MIA \u0012 HND; departure 2026-10-26T19:21:00; arrival 2026-10-28T00:55:00 | British Airways: HND \u0012 MIA; departure 2026-11-01T00:00:00; arrival 2026-11-01T02:34:00. Offer ID: off_0000BB5ZKegijyRC8tLX1A",
      "cost_usd_total": 2496.9,
      "is_estimate": true
    }
  ],
  "notes": "Duffel test-mode sample results, not real market quotes. Costs cover all requested adult travelers and both flight directions. Additional paid services are excluded. Choose one flight option; do not sum them."
}


In [ ]:
for index, option in enumerate(transport_plan.options, start=1):
    print(f"\nOption {index}")
    print(option.description)
    print(f"Whole-group round-trip cost: ${option.cost_usd_total:,.2f}")
    print(f"Sample/estimate: {option.is_estimate}")

print("\nNotes:", transport_plan.notes)


Option 1
DUFFEL TEST SAMPLE: British Airways: MIA  HND; departure 2026-10-26T19:21:00; arrival 2026-10-28T00:55:00 | British Airways: HND  MIA; departure 2026-11-01T00:00:00; arrival 2026-11-01T02:34:00. Offer ID: off_0000BB5ZKegijyRC8tLX1A
Whole-group round-trip cost: $2,496.90
Sample/estimate: True

Notes: Duffel test-mode sample results, not real market quotes. Costs cover all requested adult travelers and both flight directions. Additional paid services are excluded. Choose one flight option; do not sum them.


In [ ]:
mock_transport = await search_flights_raw(
    origin=transport_trip.origin,
    destination=transport_trip.destination,
    start_date=transport_trip.start_date,
    end_date=transport_trip.end_date,
    travelers=transport_trip.travelers,
    force_mock=True,
)

assert mock_transport["source"] == "mock"
assert len(mock_transport["options"]) == 3

expected_total = 800.0 * transport_trip.travelers
assert (
    mock_transport["options"][0]["cost_usd_total"]
    == expected_total
)

print("Mock fallback test passed.")
print(json.dumps(mock_transport, indent=2))

Mock fallback test passed.
{
  "source": "mock",
  "options": [
    {
      "mode": "flight",
      "description": "MOCK: Sample economy option A. Round trip MIA \u2192 HND \u2192 MIA. Outbound: 2026-10-26. Return: 2026-11-01. 3 adult travelers. No real airline or schedule is represented.",
      "cost_usd_total": 2400.0,
      "is_estimate": true
    },
    {
      "mode": "flight",
      "description": "MOCK: Sample economy option B. Round trip MIA \u2192 HND \u2192 MIA. Outbound: 2026-10-26. Return: 2026-11-01. 3 adult travelers. No real airline or schedule is represented.",
      "cost_usd_total": 2850.0,
      "is_estimate": true
    },
    {
      "mode": "flight",
      "description": "MOCK: Sample economy option C. Round trip MIA \u2192 HND \u2192 MIA. Outbound: 2026-10-26. Return: 2026-11-01. 3 adult travelers. No real airline or schedule is represented.",
      "cost_usd_total": 3300.0,
      "is_estimate": true
    }
  ],
  "notes": "Synthetic demo data only. These are not r